# NOTEBOOK: 04 — LLM Judge

- **Layer:** Silver
- **Purpose:** Validate LLM extraction results using LLM-as-Judge. Judge independently classifies each product and flags disagreements.
- **Inputs:** `main.techmart_silver.llm_extracted`
- **Outputs:** `main.techmart_silver.taxonomy`
- **Notes:** Returns `judge_taxonomy`, `judge_approved` and `judge_reason` per record. All runs traced in MLflow.
- **Author:** Maira Tavares

#  0. Imports and Configuration

In [0]:
import mlflow
import time
import json
from pyspark.sql import functions as F
import sys
import os
from pathlib import Path


In [0]:
REPO_ROOT = "/Workspace" + str(Path(dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()).parent.parent)

from utils.config import (
    # Source and target tables
    LLM_EXTRACTED,
    SILVER_TAXONOMY,
    # LLM model configuration
    LLM_MODEL,
    LLM_API_URL,
    LLM_PROVIDER,
    LLM_MAX_RETRIES,
    LLM_RETRY_DELAY,
    LLM_TIMEOUT,
    # Prompt configuration
    PROMPT_FOLDER,
    PROMPT_JUDGE,
    PROMPT_VERSION,
    APPROVED_TAXONOMY,
    # MLflow
    MLFLOW_EXPERIMENT_NAME,
    # Runtime functions
    get_api_key
)
from utils.states import JudgeResult
from utils.llm_utils import call_llm, load_prompt, load_prompt_template_raw


In [0]:
PROMPTS_DIR       = Path(REPO_ROOT) / PROMPT_FOLDER
LLM_API_KEY       = get_api_key(dbutils)
MLFLOW_EXPERIMENT = (
    f"/Users/"
    f"{dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()}"
    f"/{MLFLOW_EXPERIMENT_NAME}"
)

In [0]:
# ── Summary ───────────────────────────────────────────────────
print("=" * 55)
print("NOTEBOOK 04 — LLM JUDGE")
print("=" * 55)
print(f"  {'Repo root':<20} : {REPO_ROOT}")
print(f"  {'Prompts dir':<20} : {PROMPTS_DIR}")
print(f"  {'Prompt version':<20} : {PROMPT_VERSION}")
print(f"  {'Model':<20} : {LLM_MODEL}")
print(f"  {'Provider':<20} : {LLM_PROVIDER}")
print(f"  {'Max retries':<20} : {LLM_MAX_RETRIES}")
print(f"  {'Retry delay':<20} : {LLM_RETRY_DELAY}s")
print(f"  {'MLflow experiment':<20} : {MLFLOW_EXPERIMENT}")
print(f"  {'Source table':<20} : {LLM_EXTRACTED}")
print(f"  {'Target table':<20} : {SILVER_TAXONOMY}")
print("=" * 55)
print("✅ Ready to run")

# 1. Load Data

In [0]:
# Read LLM extracted table
# Only judges records that were successfully extracted.
# Failed extractions are excluded — they have no name/brand/sub_category to validate.

df_extracted = spark.table(LLM_EXTRACTED).filter(
    F.col("llm_status") == "success"
)

print(f"Records to judge: {df_extracted.count()}")
display(df_extracted.select(
    "product_id",
    "product_description",
    "extracted_name",
    "extracted_brand",
    "extracted_sub_category"
))

# 2. Functions

In [0]:
# Judge function
def judge_product(
    product_description: str,
    name               : str,
    brand              : str,
    sub_category       : str
) -> dict:
    """
    TechMart-specific LLM Judge logic.

    Two-step validation:
    1. Independently classifies the product using description + extracted name
    2. Compares its classification with the extractor's sub_category

    Uses Jinja2 templates for prompt rendering, call_llm for the API call
    with exponential backoff retry, and Pydantic for response validation.

    Args:
        product_description: Original product description from Silver layer.
        name               : Product name extracted by 03_llm_extraction.
        brand              : Brand extracted by 03_llm_extraction.
        sub_category       : Sub-category extracted by 03_llm_extraction.

    Returns:
        dict with judge_taxonomy, judge_approved, reason + observability metrics.

    Raises:
        ValueError: if all retry attempts are exhausted.
    """
    system_prompt = load_prompt(
        PROMPTS_DIR, PROMPT_JUDGE,
        role="system",
        approved_taxonomy=APPROVED_TAXONOMY
    )
    user_prompt = load_prompt(
        PROMPTS_DIR, PROMPT_JUDGE,
        role="user",
        product_description=product_description,
        name=name,
        brand=brand,
        sub_category=sub_category
    )

    result = call_llm(
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt}
        ],
        api_key      = LLM_API_KEY,
        api_url      = LLM_API_URL,
        model        = LLM_MODEL,
        max_tokens   = 100,
        max_retries  = LLM_MAX_RETRIES,
        retry_delay  = LLM_RETRY_DELAY,
        timeout      = LLM_TIMEOUT,
        output_model = JudgeResult
    )

    judge_result = result["validated_output"]

    return {
        "judge_taxonomy" : judge_result.judge_taxonomy,
        "judge_approved" : judge_result.judge_approved,
        "reason"         : judge_result.reason,
        "input_tokens"   : result["input_tokens"],
        "output_tokens"  : result["output_tokens"],
        "latency"        : result["latency_seconds"]
    }

# 3. Run llm judge with MLflow tracing

In [0]:
# Run judge with MLflow tracing
mlflow.set_experiment(MLFLOW_EXPERIMENT)

records     = df_extracted.collect()
all_records = []

with mlflow.start_run(run_name=f"judge_{PROMPT_VERSION}"):

    # Log judge prompt template as artifact
    mlflow.log_text(
        load_prompt_template_raw(PROMPTS_DIR, PROMPT_JUDGE),
        PROMPT_JUDGE
    )

    mlflow.log_param("prompt_version", PROMPT_VERSION)
    mlflow.log_param("model",          LLM_MODEL)
    mlflow.log_param("provider",       LLM_PROVIDER)
    mlflow.log_param("total_records",  len(records))

    approved_count   = 0
    flagged_count    = 0
    total_input_tok  = 0
    total_output_tok = 0
    total_latency    = 0.0

    for row in records:
        product_id          = row["product_id"]
        product_description = row["product_description"]
        name                = row["extracted_name"]
        brand               = row["extracted_brand"]
        sub_category        = row["extracted_sub_category"]

        try:
            result = judge_product(
                product_description = product_description,
                name                = name,
                brand               = brand,
                sub_category        = sub_category
            )

            total_input_tok  += result["input_tokens"]
            total_output_tok += result["output_tokens"]
            total_latency    += result["latency"]

            if result["judge_approved"]:
                approved_count += 1
            else:
                flagged_count += 1

            all_records.append({
                "product_id"            : str(product_id),
                "product_description"   : product_description,
                "extracted_name"        : name,
                "extracted_brand"       : brand,
                "extracted_sub_category": sub_category,
                "judge_taxonomy"        : result["judge_taxonomy"],
                "judge_approved"        : str(result["judge_approved"]),
                "judge_reason"          : result["reason"],
                "prompt_version"        : PROMPT_VERSION
            })

        except Exception as e:
            flagged_count += 1
            all_records.append({
                "product_id"            : str(product_id),
                "product_description"   : product_description,
                "extracted_name"        : name,
                "extracted_brand"       : brand,
                "extracted_sub_category": sub_category,
                "judge_taxonomy"        : None,
                "judge_approved"        : "False",
                "judge_reason"          : f"Judge call failed: {str(e)}",
                "prompt_version"        : PROMPT_VERSION
            })

        # Respect Groq free tier TPM rate limit
        time.sleep(5)

    mlflow.log_metric("approved_count",      approved_count)
    mlflow.log_metric("flagged_count",       flagged_count)
    mlflow.log_metric("approval_rate",       round(approved_count / len(records), 3))
    mlflow.log_metric("total_input_tokens",  total_input_tok)
    mlflow.log_metric("total_output_tokens", total_output_tok)
    mlflow.log_metric("total_tokens",        total_input_tok + total_output_tok)
    mlflow.log_metric("avg_latency_seconds", round(total_latency / len(records), 3))

print(f"✅ Judging complete")
print(f"   Approved : {approved_count}")
print(f"   Flagged  : {flagged_count}")

# 4. Save and Validate

In [0]:
# Write to Delta and validate
(spark.createDataFrame(all_records)
    .select(
        "product_id",
        "product_description",
        "extracted_brand",
        "extracted_name",
        "extracted_sub_category",
        "judge_taxonomy",
        "judge_approved",
        "judge_reason",
        "prompt_version"
    )
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TAXONOMY))

print(f"✅ Written : {SILVER_TAXONOMY}")
print(f"   Rows    : {spark.table(SILVER_TAXONOMY).count()}")

In [0]:
# Validation summary
print("=" * 55)
print("LLM JUDGE — VALIDATION SUMMARY")
print("=" * 55)
print(f"  Total records : {spark.table(SILVER_TAXONOMY).count()}")
print(f"  Approved      : {spark.table(SILVER_TAXONOMY).filter(F.col('judge_approved') == 'True').count()}")
print(f"  Flagged       : {spark.table(SILVER_TAXONOMY).filter(F.col('judge_approved') == 'False').count()}")
print("=" * 55)
print("✅ LLM Judge complete")
display(spark.table(SILVER_TAXONOMY))